# Week 3-4: Structured Content Archetype Clustering
**FlyRank Applied Search Intelligence ML Internship**

### Objective:
1. Group enterprise content assets into distinct behavioral performance archetypes using unsupervised learning.
2. Map mathematical cluster centroids to business action playbooks (Protect, Refresh, Metadata Polish, Prune/Consolidate).
3. Visualize the 2D cluster manifold using Principal Component Analysis (PCA).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

df = pd.read_csv('../data/content_refresh_anonymized.csv')

# Select clustering features
cluster_cols = ['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update']
X = df[cluster_cols].fillna(df[cluster_cols].median())

# Log-transform skewed distributions & scale
X_scaled = StandardScaler().fit_transform(np.log1p(np.maximum(X, 0)))

# Fit KMeans (k=4)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['archetype_cluster'] = kmeans.fit_predict(X_scaled)

centroids = df.groupby('archetype_cluster')[cluster_cols].mean()
print("Cluster Centroids (Raw Means):
", centroids)


## Archetype Mapping & Business Action Matrix
We sort clusters by search equity (impressions & clicks) and assign concrete business action strategies.


In [ ]:
sorted_by_imp = centroids['impressions_90d'].sort_values(ascending=False).index.tolist()

cluster_names = {
    sorted_by_imp[0]: 'Powerhouse Core (Action: Protect & Maintain)',
    sorted_by_imp[1]: 'Decaying Traffic Stars (Action: High-Priority Refresh)',
    sorted_by_imp[2]: 'High-Impression CTR Gap (Action: Snippet / Metadata Fix)',
    sorted_by_imp[3]: 'Zombie Longtail (Action: Consolidate / Prune)'
}

df['archetype_name'] = df['archetype_cluster'].map(cluster_names)
print(df['archetype_name'].value_counts())


In [ ]:
# PCA Decomposition for 2D Manifold Visualization
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df['pca_1'] = pca_coords[:, 0]
df['pca_2'] = pca_coords[:, 1]

plt.figure(figsize=(10, 6.5))
palette = {
    'Powerhouse Core (Action: Protect & Maintain)': '#2563eb',
    'Decaying Traffic Stars (Action: High-Priority Refresh)': '#f59e0b',
    'High-Impression CTR Gap (Action: Snippet / Metadata Fix)': '#10b981',
    'Zombie Longtail (Action: Consolidate / Prune)': '#ef4444'
}

sns.scatterplot(
    data=df.sample(3000, random_state=42),
    x='pca_1', y='pca_2', hue='archetype_name',
    alpha=0.6, s=35, palette=palette
)
plt.title('Content Performance Archetype Manifold (PCA)', fontsize=13, fontweight='bold')
plt.xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend(title='Assigned Archetype', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


### Week 3-4 Summary:
Clustering segments unstructured enterprise content libraries into 4 actionable buckets:
1. **Powerhouse Core**: High equity, strong position (<10) — protect technical health, do not disrupt.
2. **Decaying Traffic Stars**: High historical equity experiencing rank drift — highest ROI target for editorial refresh.
3. **High-Impression CTR Gap**: Good visibility (positions 8–18) but low CTR — target for title tag / meta description / schema markup testing.
4. **Zombie Longtail**: Stale pages with negligible demand — candidates for 301 consolidation or pruning to preserve crawl budget.
